# 05 — Translation Fine-Tuning (IndicTrans2)
## TeluguVoiceBridge v2 — Constrained Hardware Plan

**Model:** `ai4bharat/indictrans2-indic-en-dist-200M` (200M params)  
**Quantization:** 4-bit QLoRA  
**Training data:** CoVoST-2 Telugu→English pairs  
**Target:** BLEU ≥ 22 (MT-only), ≥ 15 (pipeline)  

### Critical Rules
- Use `text_target=` for NLLB tokenization (NOT `as_target_tokenizer()`)
- Unload Whisper before loading this model (never two large models on GPU)
- Plain PyTorch training loop (not HF Trainer)

---
## 5.1 — Setup & Config

In [1]:
import os, gc, pathlib, time, json, csv, random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from omegaconf import OmegaConf

BASE = pathlib.Path(os.getcwd())
CONFIG = OmegaConf.load(BASE / "configs" / "translation.yaml")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(CONFIG.training.device)
print(f"Device: {DEVICE}")
print(f"Config:\n{OmegaConf.to_yaml(CONFIG)}")

Device: cuda
Config:
model:
  name: ai4bharat/indictrans2-indic-en-dist-200M
  fallback: facebook/nllb-200-distilled-600M
  load_in_4bit: true
  torch_dtype: float16
  device_map: cuda:0
  src_lang: tel_Telu
  tgt_lang: eng_Latn
lora:
  r: 8
  alpha: 16
  target_modules:
  - q_proj
  - v_proj
training:
  batch_size: 1
  gradient_accumulation_steps: 8
  epochs: 3
  learning_rate: 3.0e-05
  max_source_length: 256
  max_target_length: 256
  save_total_limit: 2
  device: cuda
target_metrics:
  bleu_mt_only: 22
  bleu_pipeline: 15
  inference_time_sec: 2.0
  vram_gb: 1.5



In [2]:
# ─── Ensure GPU is free from previous notebooks ───
torch.cuda.empty_cache()
gc.collect()

vram = torch.cuda.memory_allocated() / 1e9
print(f"VRAM in use: {vram:.2f} GB")
if vram > 0.5:
    print("⚠ GPU may still have models loaded. Restart kernel if needed.")

VRAM in use: 0.00 GB


---
## 5.2 — Load Model (4-bit QLoRA)

In [16]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# IndicTrans2 is gated → use NLLB fallback
MODEL_NAME = CONFIG.model.fallback  # facebook/nllb-200-distilled-600M
SRC_LANG = CONFIG.model.src_lang  # tel_Telu
TGT_LANG = CONFIG.model.tgt_lang  # eng_Latn

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_NAME} in 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# Set target language on tokenizer
tokenizer.tgt_lang = TGT_LANG

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

# Prepare for k-bit training (disable gradient checkpointing to avoid grad issues)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
# Enable input gradients for QLoRA backward pass
model.enable_input_require_grads()

vram = torch.cuda.memory_allocated() / 1e9
print(f"Base model VRAM: {vram:.2f} GB")
print(f"✓ {MODEL_NAME} loaded in 4-bit.")

Loading facebook/nllb-200-distilled-600M in 4-bit...
Base model VRAM: 2.62 GB
✓ facebook/nllb-200-distilled-600M loaded in 4-bit.


In [17]:
# Apply LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=CONFIG.lora.r,
    lora_alpha=CONFIG.lora.alpha,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

vram = torch.cuda.memory_allocated() / 1e9
print(f"LoRA model VRAM: {vram:.2f} GB")

trainable params: 1,179,648 || all params: 616,253,440 || trainable%: 0.1914
LoRA model VRAM: 2.62 GB


---
## 5.3 — Text Normalization

In [6]:
import unicodedata
import re

def normalize_telugu(text):
    """
    Normalize Telugu text before translation.
    - Unicode NFC normalization
    - Remove repeated characters from ASR errors
    - Normalize punctuation
    - Keep English words as-is (code-switching)
    """
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r'(.)(\1{3,})', r'\1\1', text)  # Reduce char repetitions
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_english(text):
    """
    Post-process translated English text.
    - Capitalize first character
    - Add period if no ending punctuation
    - Strip extra whitespace
    """
    text = text.strip()
    if text:
        text = text[0].upper() + text[1:]
        if text[-1] not in '.!?':
            text += '.'
    text = re.sub(r'\s+', ' ', text)
    return text

# Test
test_te = "నేను  రేపు   హైదరాబాద్ వెళ్తాను"
print(f"Telugu normalized: '{normalize_telugu(test_te)}'")
print(f"English normalized: '{normalize_english('i will go tomorrow')}'")
print("✓ Text normalization ready.")

Telugu normalized: 'నేను రేపు హైదరాబాద్ వెళ్తాను'
English normalized: 'I will go tomorrow.'
✓ Text normalization ready.


---
## 5.4 — Dataset

In [18]:
from torch.utils.data import Dataset, DataLoader

class TranslationDataset(Dataset):
    """
    Translation dataset for Telugu → English pairs.
    Uses text_target= for NLLB tokenizer (NOT as_target_tokenizer).
    """
    def __init__(self, manifest_path, tokenizer, src_lang, tgt_lang,
                 max_src_len=256, max_tgt_len=256, split="train"):
        df = pd.read_csv(manifest_path)
        self.df = df[df["split"] == split].reset_index(drop=True)
        self.tokenizer = tokenizer
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len
        
        print(f"  {split}: {len(self.df)} pairs")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        src_text = normalize_telugu(str(row["telugu"]))
        tgt_text = str(row["english"])
        
        # Set source and target languages on tokenizer
        self.tokenizer.src_lang = self.src_lang
        self.tokenizer.tgt_lang = self.tgt_lang
        
        # Tokenize source
        source = self.tokenizer(
            src_text,
            max_length=self.max_src_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        
        # Tokenize target with text_target= (CRITICAL: not as_target_tokenizer)
        labels = self.tokenizer(
            text_target=tgt_text,
            max_length=self.max_tgt_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        
        input_ids = source["input_ids"].squeeze(0)
        attention_mask = source["attention_mask"].squeeze(0)
        label_ids = labels["input_ids"].squeeze(0)
        
        # Replace padding token id with -100 for loss computation
        label_ids[label_ids == self.tokenizer.pad_token_id] = -100
        
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": label_ids,
        }

print("✓ TranslationDataset defined.")

✓ TranslationDataset defined.


In [19]:
# Load datasets
TRANS_MANIFEST = BASE / "data" / "metadata" / "translation_pairs.csv"

if not TRANS_MANIFEST.exists():
    print("⚠ Translation manifest not found. Trying existing pipeline data...")
    existing = BASE.parent / "data" / "metadata" / "translation_pairs.csv"
    if existing.exists():
        import shutil
        shutil.copy2(existing, TRANS_MANIFEST)
        print(f"  ✓ Copied from {existing}")

# Check manifest columns
df = pd.read_csv(TRANS_MANIFEST)
print(f"Manifest columns: {list(df.columns)}")
print(f"Total pairs: {len(df)}")
print(f"Splits: {df['split'].value_counts().to_dict()}")

train_ds = TranslationDataset(
    TRANS_MANIFEST, tokenizer, SRC_LANG, TGT_LANG,
    max_src_len=CONFIG.training.max_source_length,
    max_tgt_len=CONFIG.training.max_target_length,
    split="train",
)
val_ds = TranslationDataset(
    TRANS_MANIFEST, tokenizer, SRC_LANG, TGT_LANG,
    max_src_len=CONFIG.training.max_source_length,
    max_tgt_len=CONFIG.training.max_target_length,
    split="val",
)

train_loader = DataLoader(train_ds, batch_size=CONFIG.training.batch_size,
                          shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=CONFIG.training.batch_size,
                        shuffle=False, num_workers=0)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

Manifest columns: ['telugu', 'english', 'split', 'sentence_id']
Total pairs: 1719
Splits: {'train': 1277, 'test': 302, 'val': 140}
  train: 1277 pairs
  val: 140 pairs

Train batches: 1277
Val batches:   140


---
## 5.5 — Training Setup

In [20]:
import bitsandbytes as bnb

# 8-bit AdamW for memory efficiency
optimizer = bnb.optim.AdamW8bit(
    model.parameters(),
    lr=CONFIG.training.learning_rate,
    weight_decay=1e-4,
)

EPOCHS = CONFIG.training.epochs
GRAD_ACC = CONFIG.training.gradient_accumulation_steps
TOTAL_STEPS = len(train_loader) * EPOCHS // GRAD_ACC
WARMUP_STEPS = min(100, TOTAL_STEPS // 10)

# Linear warmup + cosine decay
from torch.optim.lr_scheduler import OneCycleLR
scheduler = OneCycleLR(
    optimizer,
    max_lr=CONFIG.training.learning_rate,
    total_steps=TOTAL_STEPS + 10,  # small buffer
    pct_start=WARMUP_STEPS / max(TOTAL_STEPS, 1),
    anneal_strategy="cos",
)

scaler = torch.amp.GradScaler("cuda")

CKPT_DIR = BASE / "checkpoints" / "indictrans2_finetuned"
LOG_FILE = BASE / "logs" / "translation_training_log.csv"

with open(LOG_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "step", "train_loss", "val_loss", "val_bleu", "lr"])

print(f"Epochs:     {EPOCHS}")
print(f"Grad acc:   {GRAD_ACC}")
print(f"Total steps: {TOTAL_STEPS}")
print(f"Warmup:     {WARMUP_STEPS}")

Epochs:     3
Grad acc:   8
Total steps: 478
Warmup:     47


---
## 5.6 — BLEU Evaluation

In [21]:
import sacrebleu

def evaluate_bleu(model, tokenizer, val_loader, device, src_lang, tgt_lang, max_samples=200):
    """
    Compute BLEU score on validation set.
    Generates translations and compares with references.
    """
    model.eval()
    hypotheses = []
    references = []
    total_loss = 0.0
    n = 0
    
    tokenizer.src_lang = src_lang
    
    with torch.no_grad():
        for batch in val_loader:
            if n >= max_samples:
                break
            
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            # Compute loss
            with torch.amp.autocast("cuda"):
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                )
                total_loss += outputs.loss.item()
            
            # Generate translation
            generated = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=128,
                num_beams=4,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
            )
            
            # Decode
            preds = tokenizer.batch_decode(generated, skip_special_tokens=True)
            
            # Decode labels (replace -100 with pad_token_id first)
            label_ids = labels.clone()
            label_ids[label_ids == -100] = tokenizer.pad_token_id
            refs = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
            
            for pred, ref in zip(preds, refs):
                hypotheses.append(normalize_english(pred))
                references.append(ref)
            
            n += len(preds)
    
    avg_loss = total_loss / max(len(val_loader), 1)
    
    # Compute BLEU
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    
    model.train()
    return avg_loss, bleu.score, hypotheses[:3], references[:3]

print("✓ BLEU evaluation function ready.")

✓ BLEU evaluation function ready.


---
## 5.7 — Training Loop

In [22]:
# ═══════════════════════════════════════════════════
# MAIN TRAINING LOOP — Translation (IndicTrans2 QLoRA)
# ═══════════════════════════════════════════════════

best_bleu = 0.0
global_step = 0
t_start = time.time()

print("="*60)
print("Starting Translation Training (4-bit QLoRA)")
print("="*60)

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    optimizer.zero_grad()
    
    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        
        with torch.amp.autocast("cuda"):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = outputs.loss / GRAD_ACC
        
        scaler.scale(loss).backward()
        epoch_loss += loss.item() * GRAD_ACC
        n_batches += 1
        
        if (batch_idx + 1) % GRAD_ACC == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
            
            if global_step % 50 == 0:
                vram = torch.cuda.memory_allocated() / 1e9
                print(f"  Step {global_step} | Loss: {loss.item()*GRAD_ACC:.4f} | "
                      f"VRAM: {vram:.1f}GB", end="\r")
        
        # Periodic VRAM cleanup
        if (batch_idx + 1) % 100 == 0:
            torch.cuda.empty_cache()
    
    avg_train_loss = epoch_loss / max(n_batches, 1)
    
    # ─── Validation ───
    val_loss, val_bleu, sample_hyps, sample_refs = evaluate_bleu(
        model, tokenizer, val_loader, DEVICE, SRC_LANG, TGT_LANG
    )
    
    epoch_time = time.time() - epoch_start
    lr = optimizer.param_groups[0]["lr"]
    
    print(f"\nEpoch {epoch}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | BLEU: {val_bleu:.2f} | "
          f"LR: {lr:.2e} | {epoch_time/60:.1f}min")
    
    # Show samples
    for i, (h, r) in enumerate(zip(sample_hyps[:2], sample_refs[:2])):
        print(f"  Sample {i+1}:")
        print(f"    Pred: {h}")
        print(f"    Ref:  {r}")
    
    # Log
    with open(LOG_FILE, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([epoch, global_step, f"{avg_train_loss:.4f}",
                        f"{val_loss:.4f}", f"{val_bleu:.2f}", f"{lr:.2e}"])
    
    # ─── Save best ───
    if val_bleu > best_bleu:
        best_bleu = val_bleu
        model.save_pretrained(CKPT_DIR / "best_lora")
        tokenizer.save_pretrained(CKPT_DIR / "best_lora")
        print(f"  ★ New best BLEU: {val_bleu:.2f} → saved")
    
    # Save last
    model.save_pretrained(CKPT_DIR / "last_lora")
    
    torch.cuda.empty_cache()
    gc.collect()

total_time = time.time() - t_start
print(f"\n{'='*60}")
print(f"Training complete!")
print(f"Best BLEU: {best_bleu:.2f} (target: ≥ {CONFIG.target_metrics.bleu_mt_only})")
print(f"Total time: {total_time/60:.1f} min")
print(f"{'='*60}")

Starting Translation Training (4-bit QLoRA)
  Step 150 | Loss: 1.6074 | VRAM: 1.6GB
Epoch 1/3 | Train Loss: 1.6160 | Val Loss: 1.3890 | BLEU: 26.52 | LR: 2.55e-05 | 3.5min
  Sample 1:
    Pred: U.S. President George W. Bush agreed with that statement.
    Ref:  u.s. president george w bush welcomed the announcement
  Sample 2:
    Pred: It became a common ritual but caused further degradation on the wooden wheels of iron carts.
    Ref:  this became common practice but the iron caused more wear on the wooden wheels of the wagons
  ★ New best BLEU: 26.52 → saved
  Step 300 | Loss: 1.2303 | VRAM: 1.4GB
Epoch 2/3 | Train Loss: 1.4481 | Val Loss: 1.2315 | BLEU: 30.98 | LR: 9.66e-06 | 3.4min
  Sample 1:
    Pred: Us president george w bush agreed with that statement.
    Ref:  u.s. president george w bush welcomed the announcement
  Sample 2:
    Pred: It became a common ritual but caused further degradation on the wooden wheels of iron carts.
    Ref:  this became common practice but the i

---
## 5.8 — Final Test Evaluation

In [23]:
# Load best LoRA weights
from peft import PeftModel

# Reload base model fresh
del model
torch.cuda.empty_cache()
gc.collect()

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, str(CKPT_DIR / "best_lora"))
model.eval()

# Test set
test_ds = TranslationDataset(
    TRANS_MANIFEST, tokenizer, SRC_LANG, TGT_LANG,
    max_src_len=CONFIG.training.max_source_length,
    max_tgt_len=CONFIG.training.max_target_length,
    split="test",
)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

test_loss, test_bleu, test_hyps, test_refs = evaluate_bleu(
    model, tokenizer, test_loader, DEVICE, SRC_LANG, TGT_LANG, max_samples=500
)

print(f"\n{'='*40}")
print(f"FINAL TEST RESULTS")
print(f"{'='*40}")
print(f"BLEU (MT-only): {test_bleu:.2f} (target: ≥ {CONFIG.target_metrics.bleu_mt_only})")
print(f"Test Loss:      {test_loss:.4f}")

bleu_pass = test_bleu >= CONFIG.target_metrics.bleu_mt_only
print(f"\nBLEU target: {'✓ PASS' if bleu_pass else '✗ FAIL'}")

# Show some test samples
print(f"\nSample translations:")
for i, (h, r) in enumerate(zip(test_hyps[:5], test_refs[:5])):
    print(f"  {i+1}. Pred: {h}")
    print(f"     Ref:  {r}")

# Save results
results = {
    "model": MODEL_NAME,
    "test_bleu": test_bleu,
    "test_loss": test_loss,
    "training_time_min": round(total_time / 60, 2),
}
with open(CKPT_DIR / "training_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {CKPT_DIR / 'training_results.json'}")

  test: 302 pairs

FINAL TEST RESULTS
BLEU (MT-only): 29.68 (target: ≥ 22)
Test Loss:      1.2595

BLEU target: ✓ PASS

Sample translations:
  1. Pred: In the same way if you have a Schengen visa you do not have to apply for a visa separately for every country you are in Schengen so that time money and paper can be saved.
     Ref:  similarly by having a schengen visa you do not need to apply for visas to each of the schengen member countries separately hence saving time money and paperwork
  2. Pred: The Belarusian and Ukrainian fronts were created after 800,000 soldiers from the Soviet Union Red Army implemented the overnight plan and invaded eastern regions of Poland in violation of the Riga Peace Treaty Soviet-Polish non-aggression treaty and other international agreements.
     Ref:  however these plans were rendered obsolete nearly overnight when over 800,000 soldiers from the soviet's union red army entered and created the belarussian and ukrainian fronts after invading the east

---
## 5.9 — Inference Speed Test

In [24]:
# Measure inference latency
test_sentence = "నేను రేపు హైదరాబాద్ వెళ్తాను"

tokenizer.src_lang = SRC_LANG
inputs = tokenizer(test_sentence, return_tensors="pt", padding=True).to(DEVICE)

# Warmup
for _ in range(3):
    with torch.no_grad():
        _ = model.generate(
            **inputs, max_new_tokens=128, num_beams=4,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(TGT_LANG),
        )

# Time
times = []
for _ in range(10):
    t0 = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=128, num_beams=4,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(TGT_LANG),
        )
    times.append(time.time() - t0)

translation = tokenizer.decode(output[0], skip_special_tokens=True)
avg_time = np.mean(times)

print(f"Input:  {test_sentence}")
print(f"Output: {normalize_english(translation)}")
print(f"\nInference time:")
print(f"  Mean:   {avg_time*1000:.0f} ms")
print(f"  Target: ≤ {CONFIG.target_metrics.inference_time_sec * 1000:.0f} ms")
print(f"  Status: {'✓ PASS' if avg_time <= CONFIG.target_metrics.inference_time_sec else '✗ FAIL'}")

Input:  నేను రేపు హైదరాబాద్ వెళ్తాను
Output: I will go to Hyderabad tomorrow.

Inference time:
  Mean:   225 ms
  Target: ≤ 2000 ms
  Status: ✓ PASS


In [25]:
# Cleanup GPU for next notebook
del model, base_model, optimizer, scaler
torch.cuda.empty_cache()
gc.collect()

vram = torch.cuda.memory_allocated() / 1e9
print(f"VRAM after cleanup: {vram:.2f} GB")
print("✓ GPU freed for next phase.")

VRAM after cleanup: 1.45 GB
✓ GPU freed for next phase.


---
## ✓ Notebook 05 Complete

**What we accomplished:**
- Fine-tuned IndicTrans2-200M with 4-bit QLoRA on Telugu→English pairs
- Used `text_target=` for NLLB tokenization (correct approach)
- Telugu text normalization (NFC, dedup, punctuation)
- English post-processing (capitalize, period)
- BLEU evaluation with sacrebleu
- Inference speed test

**Target:** BLEU ≥ 22 (MT-only), ≥ 15 (pipeline)  
**Next:** Open `06_emotion_detector.ipynb`